# 🚀 Host Ollama on Free Cloud GPU (T4 GPU + Ngrok)

> **Before running:** Go to `Runtime → Change runtime type → T4 GPU` and click Save.

Then click `Runtime → Run all` and wait for the final cell to print your public URL.

In [ ]:
# ── CELL 1: Install system dependencies (zstd is required by Ollama installer) ──
!sudo apt-get update -qq
!sudo apt-get install -y zstd curl
print("✅ System dependencies installed.")

In [ ]:
# ── CELL 2: Install Ollama ──
!curl -fsSL https://ollama.com/install.sh | sh
print("✅ Ollama installed.")

In [ ]:
# ── CELL 3: Start the Ollama server in the background ──
import subprocess
import time
import os

# Set environment variable so Ollama uses the GPU
env = os.environ.copy()
env["OLLAMA_NUM_PARALLEL"] = "4"

ollama_proc = subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=env
)
print("Starting Ollama server... waiting 8 seconds...")
time.sleep(8)

# Quick health check
import urllib.request
try:
    urllib.request.urlopen("http://localhost:11434").read()
    print("✅ Ollama server is RUNNING on http://localhost:11434")
except Exception as e:
    print(f"❌ Server not responding yet: {e} — try waiting a few more seconds and re-running this cell.")

In [ ]:
# ── CELL 4: Pull AI Models (This takes a few minutes depending on model size) ──
print("Downloading embedding model (nomic-embed-text) — needed for document indexing...")
!ollama pull nomic-embed-text
print("\nDownloading chat model (llama3.1) — needed for answering questions...")
!ollama pull llama3.1
print("\n✅ All models ready!")

In [ ]:
# ── CELL 5: Install pyngrok ──
!pip install pyngrok -q
print("✅ pyngrok installed.")

In [ ]:
# ── CELL 6: Create the Ngrok tunnel and print your public URL ──
from pyngrok import ngrok, conf

NGROK_AUTHTOKEN = "3Ijb19GUQha63ZofifikfWIxgN5_vu2V618TjgKTMEjziJPG"

conf.get_default().auth_token = NGROK_AUTHTOKEN

# Close any existing tunnels before opening a new one
ngrok.kill()

# Open a public HTTP tunnel on Ollama's port
tunnel = ngrok.connect(11434, "http")
public_url = tunnel.public_url

print("\n" + "="*60)
print("🎉  SUCCESS! YOUR CLOUD OLLAMA IS LIVE!  🎉")
print("="*60)
print("\n📋  Copy the line below into your local backend/.env file:")
print(f"\n    OLLAMA_BASE_URL={public_url}")
print("\n⚠️  Keep this tab OPEN. Closing it will stop the server.")
print("="*60)